# CICIDS2017 cleaning

Combines the raw CICIDS2017 CSV files and prepares numeric features and encoded labels.

The notebook removes the duplicate header column when identical, drops rows with non-finite or missing features, and saves the cleaned dataset and label mapping.

## Configure

Run this notebook from the `notebooks/` directory. Place the raw CSV files in `datasets/cicids2017/` first.

In [ ]:
import glob
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

DATASETS_DIR = Path("../datasets")
INPUT_DIR = DATASETS_DIR / "cicids2017"
OUTPUT_PATH = DATASETS_DIR / "cicids2017_cleaned.csv"
LABEL_MAPPING_PATH = DATASETS_DIR / "cicids2017_label_mapping.json"
LABEL_COLUMN = "Label"

## Load data

Combine the raw CSV files and inspect the resulting data.

In [ ]:
# Load the dataset
all_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

# Remove leading and trailing whitespace from column names
df.columns = df.columns.str.strip()

print(f"Rows: {len(df):,}  Columns: {df.shape[1]}")

# Display the dataframe info
df.info()

## Check duplicate columns

Drop `Fwd Header Length.1` only if it is identical to `Fwd Header Length`.

In [ ]:
# Check if the duplicate columns are identical, and if so, drop the duplicate column
if df["Fwd Header Length"].equals(df["Fwd Header Length.1"]):
    print(
        "The columns 'Fwd Header Length' and 'Fwd Header Length.1' are identical. Proceeding to drop the duplicate column."
    )
    df.drop(columns=["Fwd Header Length.1"], inplace=True)
else:
    print(
        "The columns 'Fwd Header Length' and 'Fwd Header Length.1' are not identical."
    )

Inspect the columns after the duplicate check.

In [ ]:
df.info()

## Clean numeric features

Convert all features to numeric values. Non-numeric values become missing values; labels remain unchanged.

In [ ]:
labels = df[LABEL_COLUMN]

# Convert all columns (except 'Label') to numeric, coercing errors to NaN
df = df.drop(columns=[LABEL_COLUMN]).apply(pd.to_numeric, errors="coerce")

# Reattach the 'Label' column to the dataframe
df[LABEL_COLUMN] = labels

Replace positive and negative infinity with missing values, then inspect missing values in each column.

In [ ]:
# Convert infinite feature values to missing values
feature_columns = df.columns.drop(LABEL_COLUMN)
df[feature_columns] = df[feature_columns].replace([np.inf, -np.inf], np.nan)

# Check for missing values
missing_values = df.isna().sum()

# Display columns with missing values
missing_values[missing_values > 0]

Drop rows with missing values in any feature so the saved dataset contains only finite numeric inputs for modeling.

In [ ]:
# Drop rows with missing values in any feature
df.dropna(subset=feature_columns, inplace=True)

# Check for missing values again to confirm that they have been removed
missing_values = df.isna().sum()
missing_values[missing_values > 0]

## Encode labels

Convert categorical labels to integers and retain the original-to-encoded mapping.

In [ ]:
label_encoder = LabelEncoder()

# Encode the 'Label' column using Label Encoding
df[LABEL_COLUMN] = label_encoder.fit_transform(df[LABEL_COLUMN])

# Create a mapping of the original labels to their encoded values
label_mapping = dict(
    zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
)

# Display the mapping of original labels to encoded values
print("Label Mapping:")
for original_label, encoded_value in label_mapping.items():
    print(f"\t{original_label}: {encoded_value}")

## Save outputs

Save the cleaned dataset and label mapping under `datasets/`. Running this cell overwrites existing files.

In [ ]:
# Save the cleaned dataset to a new CSV file
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size / 1e6:.1f} MB)")

# Convert the label mapping to a JSON-serializable format and save it to a JSON file
label_mapping = {key: int(value) for key, value in label_mapping.items()}

# Save the label mapping to a JSON file
with open(LABEL_MAPPING_PATH, "w") as f:
    json.dump(label_mapping, f, indent=2)
print(f"Saved {LABEL_MAPPING_PATH}")